## OBJETIVO:
Identificar cada cicatriz de fogo nos mapas anuais do MapBiomas-Fogo sem perder os trinta metros de resolução.

## ESTRATEGIA
Vetorizar os dados anuais de area queimada e assim atribuir um ID para cada poligono fechado que represente um conjunto de pixeis conectados.

##ETAPAS:
#### ETAPA 0
Exportar dados do MapBiomas-Fogo coleção 4 para o Google Drive

script: [Download das imagens anuais de area queimada.js](https://code.earthengine.google.com/81bf51ebbf55a28a2021798071376a1d)


#### ETAPA 1
Vetorizar cicatrizes e enviar de volta ao Earth Engine como uma feature collection por ano.

PASSO 0: Mosaicar as imagens do ano em um só tif usando **gdalbuildvrt** e **gdal_translate**

PASSO 1: Vetorizar ela com **gdal_poligonize.py**

PASSO 2: Atribuir uma coluna de ID decimal a cada poligono com **GeoPandas**

PASSO 3: Enviar de volta ao Earth Engine como assets

NOTEBOOK: [Vetorização das cicatrizes.ipynb](https://colab.research.google.com/drive/1CtGZsOWJY-Va1h_GknIf8MJL9Eguo_YY)


#### ETAPA 2:
Rasterizar os vetores de cicatrizes anuais com a função **ee.Image().paint()** e construir um raster stack, de uma banda por ano com o valor do pixel sendo um id unico naquele ano para cada cicatriz

script: [rasterização dos dados de identificação de cicatrizes.js](https://code.earthengine.google.com/b4c08a9601167fe6eb381d909b73736f)

#### Outros resultados:
<picture>
  <img src="https://drive.google.com/uc?id=17fqkMS7klCCDgKWx2_YqTpeoCHx2q5r2">
</picture>
Base de dados: [Google Drive](https://drive.google.com/drive/folders/1zsbCJa88UyYBokRKCrgEA_cMXI82n22K?usp=sharing)

~~~javascript
// raster stack de indentificação das cicatrizes da coleção 2 do mapbiomas-fogo (1985-2022)
var scar_id = ee.Image('projects/mapbiomas-workspace/FOGO_COL2/SUBPRODUTOS/mapbiomas-fire-collection2-annual-burned-id-v1')
// raster stack com a area de cada cicatriz da coleção 2 do mapbiomas-fogo (1985-2022)
var scar_area = ee.Image('projects/mapbiomas-workspace/FOGO_COL2/SUBPRODUTOS/mapbiomas-fire-collection2-annual-burned-area-v1')
  
~~~
~~~javascript
// raster stack de indentificação das cicatrizes da coleção 1 do mapbiomas-fogo (1985-2020)
var scar_id = ee.Image('projects/workspace-ipam/assets/FOGO/mapbiomas-fire-collection1-annual-scar_id-1')
~~~
~~~javascript
// ASSETS: vetores das cicatrizes da coleção 2 do mapbiomas-fogo (1985 à 2022)
var scar_vector_1985 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL2/mbfogo-col2-1985-v1');
var scar_vector_1986 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL2/mbfogo-col2-1986-v1');
...
var scar_vector_2019 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL2/mbfogo-col1-2021-v1');
var scar_vector_2020 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL2/mbfogo-col1-2022-v1');

~~~
~~~javascript
// ASSETS: vetores das cicatrizes da coleção 1 do mapbiomas-fogo (1985 à 2020)
var scar_vector_1985 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL1/mbfogo-col1-1985-v2');
var scar_vector_1986 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL1/mbfogo-col1-1986-v2');
...
var scar_vector_2019 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL1/mbfogo-col1-2019-v2');
var scar_vector_2020 = ee.FeatureCollection('projects/workspace-ipam/assets/FOGO/VECTOR-COL1/mbfogo-col1-2020-v2');

~~~

# DESENVOLVIMENTO DA ETAPA 1
Vetorizar cicatrizes e enviar de volta ao Earth Engine como uma feature collection por ano.

In [ ]:
# preparando o ambiente para o gdal
!apt-get install -y libgdal-dev
!apt-get install -y gdal-bin
!apt-get install python3-gdal
# preparando ambiente para o geopandas
!pip install geopandas


In [ ]:
import os
os.environ['PROJ_LIB'] = '/usr/share/proj'
os.environ['GDAL_DATA'] = '/usr/share/gdal/2.2'
os.environ['GDAL_LIBRARY_PATH'] = '/usr/lib/libgdal.so'

In [ ]:
# conectando ao google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# importando as livrarias
import os
import ee
from google.colab import auth
import subprocess
from osgeo import gdal
import geopandas as gpd
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-veraarruda')

In [ ]:
# Setar o projeto no GCP (substitua pelo seu, se necessário)
PROJECT = 'ee-veraarruda'
!gcloud config set project {PROJECT}

# Setar variável de ambiente exigida por Earth Engine
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT

# Autenticação no Earth Engine
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
import os

home = '/content/drive/MyDrive/COL10_secondary_vegetation/PARTS'

years = [
2024
]

# Lista todos os arquivos na pasta .tif
all_files_tif = os.listdir(home)
all_files_tif = list(filter(lambda f: f.endswith('.tif'), all_files_tif))

# Exibe os caminhos os arquivos
all_file_paths = [os.path.join(home, f) for f in all_files_tif]
print("Endereços completos das imagens .tif:")
for path in all_file_paths:
    print(path)


In [ ]:
for year in years:
    print('{0} PASSO 0/4: Mosaicar as imagens do ano em um só tif usando gdalbuildvrt e gdal_translate '.format(year, years.index(year), len(years) - 1))

    filtered_images_tif = list(filter(lambda f: f.find('C10_vegsecondary-') >= 0, all_files_tif))
    for i, file_name in enumerate(filtered_images_tif):
        filtered_images_tif[i] = os.path.join(home, file_name)
    files_tif_str = " ".join(filtered_images_tif)

    print(f'Arquivos filtrados para o ano {year}: {filtered_images_tif}')

    name_out_tif = '{0}/mosaicos/COL10_secondary_vegetation.tif'.format(home)
    name_out_vrt = '{0}/mosaicos/COL10_secondary_vegetation.vrt'.format(home)

    vrt_command = f'gdalbuildvrt {name_out_vrt} {files_tif_str}'
    translate_command = f'gdal_translate -a_nodata 0 -co compress=DEFLATE {name_out_vrt} {name_out_tif}'

    print(f'Executando: {vrt_command}')
    vrt_status = os.system(vrt_command)

    if vrt_status != 0:
        print(f'Erro ao criar VRT para o ano {year}')
        continue

    print(f'Executando: {translate_command}')
    translate_status = os.system(translate_command)

    if translate_status != 0:
        print(f'Erro ao criar TIF para o ano {year}')
        continue

    print(f'{year} CONCLUIDO 🎉')

print('🎉CONCLUIDO PASSO 0')

In [ ]:
# Cria a pasta se ela não existir
shapes_dir = os.path.join(home, 'shapes_anuais')
os.makedirs(shapes_dir, exist_ok=True)

for year in years:
    name_out_tif = '{0}/mosaicos/COL10_secondary_vegetation.tif'.format(home)

    # PASSO 1: Vetorização com gdal
    file_name_shp = 'mosaicos/COL10_secondary_vegetation.shp'.format(home)
    home_shp = '{0}/shapes_anuais/{1}'.format(home, file_name_shp)
    print('{0} PASSO 1/4: Vetorização com gdal_poligonize.py')

    os.system('gdal_polygonize.py {0} -b 1 -f "ESRI Shapefile" {1}'.format(name_out_tif, home_shp))
    # !gdal_polygonize.py -b 1 $name_out_tif $home_shp

    print(f'{year} CONCLUIDO 🎉')

print('🎉CONCLUIDO PASSO 1')

In [ ]:
# Cria a pasta se ela não existir
shapes_dir = os.path.join(home, 'shapes_anuais_final')
os.makedirs(shapes_dir, exist_ok=True)

for year in years:
  file_name_shp = 'mapbiomas_fire_collection41_burned_coverage_{0}-v1.shp'.format(year)
  home_shp = '{0}/shapes_anuais/{1}'.format(home, file_name_shp)

  # PASSO 2: Atribuir uma coluna de ID decimal a cada poligono com GeoPandas
  print('{0} PASSO 2/4: Atribuir uma coluna de ID decimal a cada poligono'.format(year, years.index(year), len(years) - 1))

  gdf = gpd.read_file(home_shp)
  # ID decimal de identificação da cicatriz
  gdf['id'] = range(1, len(gdf) + 1)

  new_shp = home_shp.replace("shapes_anuais", "shapes_anuais_final")

  gdf.to_file(filename=new_shp, driver='ESRI Shapefile')

  print(f'{year} CONCLUIDO 🎉')

print('🎉CONCLUIDO')

In [ ]:
for year in years:
    file_name_shp = f'mapbiomas_fire_collection41_burned_coverage_{year}-v1.shp'
    new_shp = f'{home}/shapes_anuais_final/{file_name_shp}'

  # PASSO 3: Enviar para o Google Cloud Storage
    print('{0} PASSO 3/4:  Enviar para o Google Cloud Storage'.format(year, years.index(year), len(years) - 1))

    os.system('gsutil cp {0} gs://tensorflow-fire-cerrado1/col41_vectors/{1}'.format(new_shp, file_name_shp))
    os.system('gsutil cp {0} gs://tensorflow-fire-cerrado1/col41_vectors/{1}'.format(new_shp.replace('.shp', '.dbf'), file_name_shp.replace('.shp', '.dbf')))
    os.system('gsutil cp {0} gs://tensorflow-fire-cerrado1/col41_vectors/{1}'.format(new_shp.replace('.shp', '.shx'), file_name_shp.replace('.shp', '.shx')))
    os.system('gsutil cp {0} gs://tensorflow-fire-cerrado1/col41_vectors/{1}'.format(new_shp.replace('.shp', '.prj'), file_name_shp.replace('.shp', '.prj')))

  # PASSO 4: Enviar ao Earth Engine como asset
    print(f'{year} PASSO 4/4: Enviar para o Earth Engine como asset')
    asset_id = f'projects/mapbiomas-workspace/FOGO_COL4/2_Subprodutos_col41/mapbiomas-fire-collection41-annual-burned-vectors/mbfogo-col41-{year}-v1'
    cmd_gee = f'earthengine --project {PROJECT} upload table --asset_id={asset_id} gs://tensorflow-fire-cerrado1/col41_vectors/{file_name_shp}'

    print(f'Executando comando: {cmd_gee}')
    result = subprocess.run(cmd_gee, shell=True, capture_output=True, text=True)

    if result.returncode == 0:
        print(f'{year} CONCLUÍDO ✅ Tarefa enviada com sucesso 🎉')
    else:
        print(f'{year} ERRO ❌ na submissão:')
        print('STDOUT:', result.stdout)
        print('STDERR:', result.stderr)

print('🚀 TODOS OS ANOS PROCESSADOS')